In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import pandas as pd
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


train = pd.read_csv('train_cleaned.csv')
test = pd.read_csv('test_cleaned.csv')
feature_cols = [
    "Applied_Voltage_kV", "Load_Current_A", "Ambient_Temperature_C", "Test_Duration_min",
    "Sensor_S1", "Sensor_S2", "Sensor_S3",
    "S1_missing", "S2_missing", "S3_missing", "S4_missing",
    "is_duplicate_input"]


X = train[feature_cols]
y_class = train["Validity_Label_enc"]
y_reg = train["Reference_Parameter"]

lr_f1_scores = []
lr_auc_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_class.iloc[train_idx], y_class.iloc[val_idx]

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_vl_scaled = scaler.transform(X_vl)

    clf_lr = LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000)
    clf_lr.fit(X_tr_scaled, y_tr)

    preds = clf_lr.predict(X_vl_scaled)
    probs = clf_lr.predict_proba(X_vl_scaled)[:, 1]

    f1 = f1_score(y_vl, preds)
    auc = roc_auc_score(y_vl, probs)
    lr_f1_scores.append(f1)
    lr_auc_scores.append(auc)

    print(f"Fold {fold}: F1={f1:.4f}, AUC={auc:.4f}")

print(f"\nMean F1 (default threshold): {np.mean(lr_f1_scores):.4f} (+/- {np.std(lr_f1_scores):.4f})")
print(f"Mean AUC: {np.mean(lr_auc_scores):.4f} (+/- {np.std(lr_auc_scores):.4f})")

Fold 0: F1=0.3051, AUC=0.6488
Fold 1: F1=0.3611, AUC=0.6590
Fold 2: F1=0.5098, AUC=0.6673
Fold 3: F1=0.3438, AUC=0.5812
Fold 4: F1=0.4545, AUC=0.6769

Mean F1 (default threshold): 0.3949 (+/- 0.0756)
Mean AUC: 0.6466 (+/- 0.0340)


##  This to test for the Regression


In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

lin_reg_rmse = []
lin_reg_mae = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_reg.iloc[train_idx], y_reg.iloc[val_idx]

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_vl_scaled = scaler.transform(X_vl)

    reg_lin = LinearRegression()
    reg_lin.fit(X_tr_scaled, y_tr)

    preds = reg_lin.predict(X_vl_scaled)

    rmse = np.sqrt(mean_squared_error(y_vl, preds))
    mae = mean_absolute_error(y_vl, preds)
    lin_reg_rmse.append(rmse)
    lin_reg_mae.append(mae)

    print(f"Fold {fold}: RMSE={rmse:.4f}, MAE={mae:.4f}")

print(f"\nMean RMSE: {np.mean(lin_reg_rmse):.4f} (+/- {np.std(lin_reg_rmse):.4f})")
print(f"Mean MAE: {np.mean(lin_reg_mae):.4f} (+/- {np.std(lin_reg_mae):.4f})")

Fold 0: RMSE=3.9933, MAE=3.3607
Fold 1: RMSE=4.4181, MAE=3.4900
Fold 2: RMSE=3.8817, MAE=3.2317
Fold 3: RMSE=4.4362, MAE=3.5251
Fold 4: RMSE=4.4366, MAE=3.6676

Mean RMSE: 4.2332 (+/- 0.2441)
Mean MAE: 3.4550 (+/- 0.1484)
